# Tutorial: Advanced methods for customizing download and variable extraction

In some cases, the available `file_cadence` parameter for downloading files and readers for extracting variables are not sufficient.
This can happen when files in the online repositories have irregular cadence, and when they are saved in some ASCII format, which is not readable by `pd.read_csv`. 

This tutorial will show you how the `ep.download` and `ep.extract_variables_from_files` can be customized to deal with any type of data.
We start with customizing the file cadence.

## 1. Customizing the `file_cadence` parameter

`ep.download` and `ep.extract_variables_from_files` both take a `file_cadence` argument that
controls how the requested `[start_time, end_time)` range is chunked into individual files.
Out of the box, `file_cadence` supports three literals:

- `"daily"`: one file per day
- `"monthly"`: not implemented yet
- `"single_file"`: the whole range is a single file

These cover the common case, but some data sets are published on a cadence that doesn't fit
any of them; for example weekly files, or files that show up at irregular intervals with gaps.
For these cases, `file_cadence` also accepts a **callable** `Callable[[datetime], datetime]`:
given the current chunk's start time, it returns the start time of the next chunk. Both
`ep.download` and `ep.extract_variables_from_files` call it the same way, so the same
function can be passed to both.

Here, we shows two cases:

1. A **regular but non-standard** cadence (weekly), using the LANL GPS `ns41` satellite data
   from NOAA NGDC as an example.
2. An **irregular** cadence, where the next file's date can't be computed by adding a fixed
   `timedelta` and instead needs to be looked up (e.g. from a known list of available dates).

### 1. Regular weekly cadence: LANL GPS ns41

NOAA NGDC hosts LANL GPS particle data at:

`https://www.ngdc.noaa.gov/stp/space-weather/satellite-data/satellite-systems/lanl_gps/version_v1.10r2/ns41/`

Files are named `ns41_YYMMDD_v1.10.ascii` and published exactly every 7 days, anchored at
`2000-12-10` (e.g. `..._001210_...`, `..._001217_...`, `..._001224_...`). Since the interval is
fixed, the `file_cadence` callable is a one-liner.

In [ ]:
from datetime import datetime, timedelta, timezone

import el_paso as ep

ep.setup_logging()


def weekly_cadence(curr_time: datetime) -> datetime:
    return curr_time + timedelta(days=7)


start_time = datetime(2023, 1, 1, tzinfo=timezone.utc)
end_time = datetime(2023, 1, 16, tzinfo=timezone.utc)

ep.download(
    start_time,
    end_time,
    save_path="./data",
    download_url="https://www.ngdc.noaa.gov/stp/space-weather/satellite-data/satellite-systems/lanl_gps/version_v1.10r2/ns41/",
    file_name_stem="ns41_YYMMDD_v1.10.ascii",
    file_cadence=weekly_cadence,
    method="request",
    skip_existing=True,
)

### 2. Irregular cadence: files with gaps

Some data sets aren't published on any fixed interval; instrument outages, reprocessing, or
mission phases can mean a week (or more) is simply missing. In that case `curr_time + timedelta(...)`
isn't enough, because there is no fixed step to add. Since `file_cadence` is called fresh with
`curr_time` on every iteration, it can instead look up the next *known* file date from a
pre-fetched list of available dates (e.g. scraped once from the server's directory listing, or
read from an index file the data provider publishes).

The example below simulates such a gap: weekly dates are available, except one week was skipped
(This example is just an illustration. Practically, you could also use a weekly cadence and let the download fail for the missing week).

In [ ]:
from bisect import bisect_right
from datetime import datetime, timezone

# Dates known to have a file on the server, pre-fetched once (e.g. by scraping the directory
# listing). Note the gap: 2023-01-22 is missing.
available_dates = [
    datetime(2023, 1, 1, tzinfo=timezone.utc),
    datetime(2023, 1, 8, tzinfo=timezone.utc),
    datetime(2023, 1, 15, tzinfo=timezone.utc),
    # 2023-01-22 skipped: instrument outage
    datetime(2023, 1, 29, tzinfo=timezone.utc),
    datetime(2023, 2, 5, tzinfo=timezone.utc),
]


def irregular_cadence(curr_time: datetime) -> datetime:
    """Return the next known file date strictly after curr_time.

    Falls back to a large jump once the known dates are exhausted, so the calling loop
    terminates instead of looping forever.
    """
    idx = bisect_right(available_dates, curr_time)
    if idx < len(available_dates):
        return available_dates[idx]
    return curr_time + timedelta(days=365)


irregular_start = available_dates[0]
irregular_end = datetime(2023, 2, 5, tzinfo=timezone.utc)

curr = irregular_start
chunks = []
while curr < irregular_end:
    nxt = irregular_cadence(curr)
    chunks.append((curr, min(nxt, irregular_end)))
    curr = nxt

chunks

Note that the chunk `(2023-01-15, 2023-01-29)` correctly spans the missing week instead of
producing an empty or failed chunk for `2023-01-22`. `irregular_cadence` can be passed directly
to `ep.download(..., file_cadence=irregular_cadence, ...)` and
`ep.extract_variables_from_files(..., file_cadence=irregular_cadence, ...)` exactly like
`weekly_cadence` above; only the lookup logic inside the function changes.

## Customizing the reader in `ep.extract_variables_from_files`

Some file formats might not be readable by the readers implemented in `EL_PASO`.
Especially ASCII files are sometimes written in non-standard ways.
`EL_PASO` relies on the `pd.read_csv` function to read ASCII files, which is quite powerful and can be customized by the user through the `pd_read_csv_kwargs` argument of the `ep.extract_variables_from_files` function.
If it is not possible to read the given ASCII format in this way, we can also provide a custom reader through the `custom_extractors` argument, which is a dictionary with file extensions as keys and `Callables` as values.  

### Using a custom extractor to read GPS ASCII files

The GPS ASCII format needs a custom extractor: it has no column-name header row (only a JSON
block, commented out with `#`, giving each variable's `START_COLUMN`/`DIMENSION`), and several
variables span multiple whitespace-delimited columns.
By setting the `custom_extractors` argument of the `ep.extract_variables_from_files` function, we are still able to extract the variables.
Note that here, we also have to use the custom file cadence as before.

In [ ]:
import json
from pathlib import Path

import numpy as np
from astropy import units as u
from numpy.typing import NDArray


def _parse_lanl_gps_header(file_path: str) -> dict:
    header_lines = []
    with Path(file_path).open("r") as f:
        for line in f:
            if not line.startswith("#"):
                break
            header_lines.append(line[1:])

    return json.loads("".join(header_lines))


def extract_data_from_lanl_gps_ascii(
    file_path: str, extraction_infos: tuple[ep.ExtractionInfo, ...]
) -> dict[str | int, NDArray[np.generic]]:
    """Custom extractor for LANL GPS ns41-style ASCII files."""
    header = _parse_lanl_gps_header(file_path)
    data_block = np.loadtxt(file_path, comments="#")

    data: dict[str | int, NDArray[np.generic]] = {}

    for info in extraction_infos:
        name = info.name_or_column

        if name not in header:
            msg = f"Variable {name!r} not found in header of {file_path}!"
            raise ValueError(msg)

        start_column = header[name]["START_COLUMN"]
        (dimension,) = header[name]["DIMENSION"]

        if dimension == 1:
            data[name] = data_block[:, start_column]
        else:
            data[name] = data_block[:, start_column : start_column + dimension]

    return data


extraction_infos = [
    ep.ExtractionInfo(result_key="year", name_or_column="year", unit=u.dimensionless_unscaled),
    ep.ExtractionInfo(result_key="decimal_day", name_or_column="decimal_day", unit=u.day),
    ep.ExtractionInfo(
        result_key="FEDU",
        name_or_column="electron_diff_flux",
        unit=(u.cm**2 * u.s * u.sr * u.MeV) ** (-1),
    ),
    ep.ExtractionInfo(
        result_key="Energy_FEDU",
        name_or_column="electron_diff_flux_energy",
        unit=u.MeV,
        is_time_dependent=False,
    ),
]

variables = ep.extract_variables_from_files(
    start_time,
    end_time,
    file_cadence=weekly_cadence,
    data_path="./data",
    file_name_stem="ns41_YYMMDD_v1.10.ascii",
    extraction_infos=extraction_infos,
    custom_extractors={".ascii": extract_data_from_lanl_gps_ascii},
)
variables